In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from transformers import DistilBertModel, DistilBertTokenizer
import warnings
warnings.filterwarnings('ignore')

# Конфигурация
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 16
EPOCHS = 10
MAX_LEN = 128
IMG_SIZE = 224


In [2]:
import pandas as pd

pd.set_option('display.max_rows', 1000)      # показывать все строки
pd.set_option('display.max_columns', 1000)   # показывать все столбцы
pd.set_option('display.max_colwidth', None)
train_data = pd.read_csv("train.csv", sep = ';')
test_data = pd.read_csv("Test (7).csv", sep = ';')

In [3]:
train_data['description'] = train_data['description'].fillna("None")
test_data['description'] = test_data['description'].fillna("None")

In [4]:
train_data

,id,description,label
0,814469951099,"""Когда устал и жить не хочешь,Полезно помнить в гн""",Философия и религия
1,849433210092,None,Торговля и объявления
2,852458632411,"""МИР ВАШЕМУ ДОМУ! ДОРОГИЕ ДРУЗЬЯ, ХРАНИ ВАС ГОСПОДЬ""",Философия и религия
3,860243294215,"""Альбом \""Праздничные мопсы\"" https://ok.ru/mopsyata/""",Животные
4,861555576675,Умнее некоторых людей,Животные
...,...,...,...
6396,909322079091,Депутат Госдумы Михаил Романов обратился к губерна,СМИ
6397,909332887609,"""Сколько людей, столько и идей. Наша подписчица дел""",Творчество и дизайн
6398,909333845107,В Петербурге готовят к работе 70 пунктов вакцинаци,СМИ
6399,909334888398,Фарерские острова,Путешествия


In [5]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
train_data['label']  = le.fit_transform(train_data['label'])

In [6]:
le.classes_

array(['Авто и мото', 'Благотворительные фонды', 'Блоги',
       'Государственная организация', 'Животные', 'Игры', 'Кино',
       'Компьютер и интернет', 'Кулинария', 'Культура и искусство',
       'Мода красота и здоровье', 'Музыка', 'Наука', 'Образование',
       'Путешествия', 'Развлечения и юмор', 'СМИ', 'Семья дом дети',
       'Спорт', 'Творчество и дизайн', 'Торговля и объявления',
       'Увлечения и хобби', 'Философия и религия'], dtype=object)

In [7]:
train_data

,id,description,label
0,814469951099,"""Когда устал и жить не хочешь,Полезно помнить в гн""",22
1,849433210092,None,20
2,852458632411,"""МИР ВАШЕМУ ДОМУ! ДОРОГИЕ ДРУЗЬЯ, ХРАНИ ВАС ГОСПОДЬ""",22
3,860243294215,"""Альбом \""Праздничные мопсы\"" https://ok.ru/mopsyata/""",4
4,861555576675,Умнее некоторых людей,4
...,...,...,...
6396,909322079091,Депутат Госдумы Михаил Романов обратился к губерна,16
6397,909332887609,"""Сколько людей, столько и идей. Наша подписчица дел""",19
6398,909333845107,В Петербурге готовят к работе 70 пунктов вакцинаци,16
6399,909334888398,Фарерские острова,14


In [8]:
from transformers import AutoTokenizer, AutoModel

num_classes = len(le.classes_)

# Трансформации изображений
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Даталодеры
class VKDataset(Dataset):
    def __init__(self, df, img_dir, tokenizer, is_test=False):
        self.df = df
        self.img_dir = img_dir
        self.tokenizer = tokenizer
        self.is_test = is_test
        
        if not is_test:
            self.labels = df['label'] #[categories.index(label) for label in df['label']]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, f"{row['id']}")
        
        # Загрузка изображения
        image = Image.open(img_path).convert('RGB')
        image = transform(image)
        
        # Обработка текста
        text = str(row['description']) if pd.notna(row['description']) else ''
        encoded = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        if self.is_test:
            return {
                'image': image,
                'input_ids': encoded['input_ids'].flatten(),
                'attention_mask': encoded['attention_mask'].flatten()
            }
        else:
            return {
                'image': image,
                'input_ids': encoded['input_ids'].flatten(),
                'attention_mask': encoded['attention_mask'].flatten(),
                'label': torch.tensor(self.labels[idx], dtype=torch.long)
            }

# Модель
class MultimodalModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Визуальный энкодер
        self.cnn = nn.Sequential(*list(torch.hub.load('pytorch/vision', 'resnet50', pretrained=True).children())[:-1])
        self.img_classifier = nn.Linear(2048, 512)
        
        # Текстовый энкодер
        self.bert = AutoModel.from_pretrained("sergeyzh/BERTA") #DistilBertModel.from_pretrained('distilbert-base-multilingual-cased')
        self.text_classifier = nn.Linear(768, 512)
        
        # Классификатор
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, image, input_ids, attention_mask):
        # Извлечение визуальных признаков
        img_features = self.cnn(image).squeeze()
        img_features = self.img_classifier(img_features)
        
        # Извлечение текстовых признаков
        text_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        text_features = text_outputs.last_hidden_state[:, 0, :]
        text_features = self.text_classifier(text_features)
        
        # Объединение признаков
        combined = torch.cat([img_features, text_features], dim=1)
        return self.classifier(combined)

# Подготовка данных
print("Загрузка данных...")



tokenizer = AutoTokenizer.from_pretrained("sergeyzh/BERTA")

train_dataset = VKDataset(train_data, 'Train', tokenizer)
test_dataset = VKDataset(test_data, 'Test', tokenizer, is_test=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Инициализация модели
model = MultimodalModel(num_classes).to(device)
optimizer = optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# Обучение
print("Начало обучения...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        outputs = model(images, input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f'Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}')

# Предсказание
print("Генерация предсказаний...")
model.eval()
predictions = []

with torch.no_grad():
    for batch in test_loader:
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        outputs = model(images, input_ids, attention_mask)
        _, preds = torch.max(outputs, 1)
        predictions.extend(preds.cpu().numpy())

# Сохранение результатов
submission = pd.DataFrame({
    'id': test_df['id'],
    'label': [categories[p] for p in predictions]
})
submission.to_csv('submission.csv', sep=';', index=False)
print("Результаты сохранены в submission.csv")

Загрузка данных...


Using cache found in /home/kostia/.cache/torch/hub/pytorch_vision_main


Начало обучения...


KeyboardInterrupt: 